# MNIST MLP Classifier Scaffold

This notebook prepares MNIST data and sketches an MLP classifier for MoTorch.
The criterion and training loop are intentionally pending until MoTorch has a
multiclass classification loss.

In [ ]:
import gzip
import pathlib
import struct
import urllib.request

import matplotlib.pyplot as plt
import numpy as np

import motorch as mo
import motorch.nn as nn
from motorch.data import DataLoader, TensorDataset

## Download MNIST

MNIST IDX gzip files are downloaded with `urllib.request` and cached under
`examples/data/mnist/`.

In [ ]:
MNIST_BASE_URL = "https://storage.googleapis.com/cvdf-datasets/mnist/"
MNIST_FILES = {
    "train_images": "train-images-idx3-ubyte.gz",
    "train_labels": "train-labels-idx1-ubyte.gz",
    "test_images": "t10k-images-idx3-ubyte.gz",
    "test_labels": "t10k-labels-idx1-ubyte.gz",
}

project_root = pathlib.Path.cwd()
if project_root.name == "examples":
    data_dir = project_root / "data" / "mnist"
else:
    data_dir = project_root / "examples" / "data" / "mnist"

data_dir.mkdir(parents=True, exist_ok=True)
data_dir

In [ ]:
def download_mnist_file(filename):
    destination = data_dir / filename
    if destination.exists():
        return destination

    url = MNIST_BASE_URL + filename
    print(f"Downloading {url} -> {destination}")
    urllib.request.urlretrieve(url, destination)
    return destination


mnist_paths = {
    name: download_mnist_file(filename)
    for name, filename in MNIST_FILES.items()
}
mnist_paths

## Parse IDX Files

Images are normalized to `float32` values in `[0, 1]` and flattened to 784
features. Labels remain integer class IDs.

In [ ]:
def read_idx_images(path):
    with gzip.open(path, "rb") as file:
        magic, count, rows, columns = struct.unpack(">IIII", file.read(16))
        if magic != 2051:
            raise ValueError(f"Unexpected image magic number: {magic}")

        data = np.frombuffer(file.read(), dtype=np.uint8)

    images = data.reshape(count, rows * columns)
    return images.astype(np.float32) / 255.0


def read_idx_labels(path):
    with gzip.open(path, "rb") as file:
        magic, count = struct.unpack(">II", file.read(8))
        if magic != 2049:
            raise ValueError(f"Unexpected label magic number: {magic}")

        labels = np.frombuffer(file.read(), dtype=np.uint8)

    return labels.astype(np.int64)


all_train_images = read_idx_images(mnist_paths["train_images"])
all_train_labels = read_idx_labels(mnist_paths["train_labels"])
test_images = read_idx_images(mnist_paths["test_images"])
test_labels = read_idx_labels(mnist_paths["test_labels"])

validation_size = 10_000
train_images = all_train_images[:-validation_size]
train_labels = all_train_labels[:-validation_size]
validation_images = all_train_images[-validation_size:]
validation_labels = all_train_labels[-validation_size:]

train_images.shape, validation_images.shape, test_images.shape

## Build Datasets and Loaders

This assumes `motorch.data.TensorDataset` and `motorch.data.DataLoader` are
available in the public API.

In [ ]:
batch_size = 128

train_dataset = TensorDataset(mo.tensor(train_images), mo.tensor(train_labels))
validation_dataset = TensorDataset(
    mo.tensor(validation_images),
    mo.tensor(validation_labels),
)
test_dataset = TensorDataset(mo.tensor(test_images), mo.tensor(test_labels))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

train_loader, validation_loader, test_loader

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(10, 2))
for axis, image, label in zip(axes, train_images[:5], train_labels[:5]):
    axis.imshow(image.reshape(28, 28), cmap="gray")
    axis.set_title(str(label))
    axis.axis("off")

plt.tight_layout()

## Define an MLP Skeleton

The model emits 10 logits, one per MNIST class. Training is blocked until the
separate multiclass loss exists.

In [ ]:
class MNISTMLP(nn.Module):
    def __init__(self, input_features=784, hidden_features=128, classes=10):
        super().__init__()
        self.layer_1 = nn.Linear(input_features, hidden_features)
        self.activation_1 = nn.ReLU()
        self.layer_2 = nn.Linear(hidden_features, hidden_features)
        self.activation_2 = nn.ReLU()
        self.output = nn.Linear(hidden_features, classes)

    def forward(self, x):
        x = self.activation_1(self.layer_1(x))
        x = self.activation_2(self.layer_2(x))
        return self.output(x)


model = MNISTMLP()
model

## Training Pending

Do not add a custom notebook-only loss here. Once MoTorch has a multiclass
criterion, this section can create the criterion, optimizer, train loop,
validation loop, and final test evaluation.

In [ ]:
criterion = None  # Pending a MoTorch multiclass classification loss.

# Example future shape only:
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.SGD(model.parameters(), lr=0.1)
# for images, labels in train_loader:
#     logits = model(images)
#     loss = criterion(logits, labels)
#     ...